# 🌾 IndiaMandi — Exploratory Data Analysis

**Dataset:** 172,585 daily price records from 268 APMC mandis across 16 Indian states  
**Period:** 19 Feb 2025 – 19 May 2025 (90 days)  
**Commodities:** 144 unique commodities  
**Source:** data.gov.in (Government of India Open Data Portal)

This notebook explores 5 key insights about Indian agricultural commodity pricing patterns.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Dark theme for charts
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc',
    'text.color': '#ddd',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'grid.color': '#333',
    'grid.alpha': 0.5,
    'font.size': 11,
})

# Load clean data
df = pd.read_parquet('../data/clean/mandi_prices.parquet')
os.makedirs('charts', exist_ok=True)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min().strftime('%d %b %Y')} to {df['Date'].max().strftime('%d %b %Y')}")
print(f"States: {df['State'].nunique()} | Markets: {df['Market'].nunique()} | Commodities: {df['Commodity'].nunique()}")
print(f"\nTop 10 commodities by records:")
print(df['Commodity'].value_counts().head(10).to_string())

---
## 📊 Insight 1: Which Commodities Have the Highest Price Variance?

**Question:** Which commodities show the most inconsistent pricing across mandis?

**Why it matters:** High price variance means a farmer can earn significantly more (or less) depending on which mandi they sell at. These are the commodities where choosing the right mandi matters most.

In [ ]:
# Calculate Coefficient of Variation (std / mean) for each commodity
variance = (
    df.groupby('Commodity')['Modal_Price']
    .agg(['std', 'mean', 'count'])
    .query('count >= 100')
    .assign(cv=lambda x: (x['std'] / x['mean'] * 100))
    .sort_values('cv', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(variance)))
bars = ax.barh(range(len(variance)), variance['cv'], color=colors, height=0.65)
ax.set_yticks(range(len(variance)))
ax.set_yticklabels(variance.index, fontsize=12)
ax.set_xlabel('Coefficient of Variation (%)', fontsize=13)
ax.set_title('Top 10 Commodities with Highest Price Variance Across Mandis', 
             fontsize=15, fontweight='bold', pad=15)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, variance['cv']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11, color='#ccc')

plt.tight_layout()
plt.savefig('charts/01_price_variance_top10.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** Ginger (Dry) has over 100% price variation across mandis — meaning you could find it at ₹2,000/q in one mandi and ₹6,000/q in another. Perishable vegetables (Tomato, Leafy Vegetables, Capsicum) dominate this list because transportation costs and spoilage create massive regional price gaps. This is exactly the kind of information IndiaMandi helps farmers act on.

---

## 📊 Insight 2: Cheapest States for Essential Commodities

**Question:** Which states consistently offer the lowest prices for Onion, Tomato, and Potato?

**Why it matters:** These three commodities are India's kitchen essentials. Price differences between states can be 3-5x, creating arbitrage opportunities for traders and cost savings for bulk buyers.

In [ ]:
essentials = ['Onion', 'Tomato', 'Potato']
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
palette = ['#4caf50', '#ff9800', '#f44336']

for idx, (commodity, color) in enumerate(zip(essentials, palette)):
    ax = axes[idx]
    cdf = df[df['Commodity'] == commodity]
    state_avg = cdf.groupby('State')['Modal_Price'].mean().sort_values()

    bar_colors = [color if i < 3 else '#555' for i in range(len(state_avg))]
    ax.barh(range(len(state_avg)), state_avg.values, color=bar_colors, height=0.65)
    ax.set_yticks(range(len(state_avg)))
    ax.set_yticklabels(state_avg.index, fontsize=10)
    ax.set_title(f'{commodity}', fontsize=14, fontweight='bold', color=color)
    ax.set_xlabel('Avg Price (₹/quintal)', fontsize=10)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

    for i in range(min(3, len(state_avg))):
        ax.text(state_avg.values[i] + 20, i,
                f'₹{state_avg.values[i]:,.0f}', va='center', fontsize=9, color=color, fontweight='bold')

fig.suptitle('Which States Have the Cheapest Essential Commodities?', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/02_cheapest_states_essentials.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** 
- **Onion:** Madhya Pradesh is dramatically cheapest at ₹599/q — nearly half the price of Rajasthan (₹1,110/q). Kerala is the most expensive at ~₹3,000/q.
- **Tomato:** Uttar Pradesh and Haryana lead with prices under ₹900/q. Tripura pays 5-6x more.
- **Potato:** Madhya Pradesh again cheapest at ₹391/q. Northeast states (Nagaland) pay 10x more due to transportation costs.

**Pattern:** Northern agricultural states (MP, UP, Haryana, Rajasthan) consistently offer the lowest prices. Southern and northeastern states pay significant premiums.

---

## 📊 Insight 3: Monthly Price Trends — Top 5 Commodities

**Question:** How do prices move month-over-month for the most traded commodities?

**Why it matters:** Seasonal patterns help farmers decide when to sell and traders decide when to buy. Even a 5-10% monthly swing on bulk quantities translates to significant profit/loss.

In [ ]:
top5 = df['Commodity'].value_counts().head(5).index.tolist()
fig, ax = plt.subplots(figsize=(12, 7))
colors_line = ['#4caf50', '#ff9800', '#2196f3', '#e91e63', '#9c27b0']
month_names = {2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May'}

for commodity, color in zip(top5, colors_line):
    cdf = df[df['Commodity'] == commodity]
    monthly = cdf.groupby('month')['Modal_Price'].mean()
    ax.plot(monthly.index, monthly.values, marker='o', linewidth=2.5,
            markersize=8, label=commodity, color=color)

ax.set_xticks(list(month_names.keys()))
ax.set_xticklabels(list(month_names.values()), fontsize=12)
ax.set_ylabel('Avg Modal Price (₹/quintal)', fontsize=12)
ax.set_xlabel('Month (2025)', fontsize=12)
ax.set_title('Monthly Price Trends — Top 5 Commodities (Feb–May 2025)', 
             fontsize=15, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=11, framealpha=0.3)
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

plt.tight_layout()
plt.savefig('charts/03_monthly_trends_top5.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** 
- **Bhindi (Ladies Finger)** is the most expensive among the top 5, trending upward from ₹2,600 to ₹2,650/q — typical as summer increases demand.
- **Brinjal** shows a steady upward trend (~₹2,000 → ₹2,060/q), possibly reflecting supply tightening.
- **Tomato, Onion, Potato** remain relatively stable in the ₹1,400–1,700 range — these are staples with more regulated supply chains.
- The gap between the cheapest (Tomato ~₹1,440/q) and most expensive (Bhindi ~₹2,650/q) in the top 5 is nearly 2x.

---

## 📊 Insight 4: Price Distribution — How Wide Is the Spread?

**Question:** How much price variation exists within each commodity across all mandis?

**Why it matters:** A tight box means prices are consistent nationally. A wide box means there are big arbitrage opportunities — buy cheap in one mandi, sell high in another.

In [ ]:
top8 = df['Commodity'].value_counts().head(8).index.tolist()
box_data = df[df['Commodity'].isin(top8)]

fig, ax = plt.subplots(figsize=(14, 7))
bp = ax.boxplot(
    [box_data[box_data['Commodity'] == c]['Modal_Price'].values for c in top8],
    tick_labels=top8,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color='#ff9800', linewidth=2),
    whiskerprops=dict(color='#888'),
    capprops=dict(color='#888'),
)

box_colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top8)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor('#666')

ax.set_ylabel('Modal Price (₹/quintal)', fontsize=12)
ax.set_title('Price Distribution — Top 8 Commodities Across All Mandis', 
             fontsize=15, fontweight='bold', pad=15)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.xticks(rotation=25, ha='right')

plt.tight_layout()
plt.savefig('charts/04_price_spread_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:**
- **Banana** has the widest price spread (₹800 to ₹7,500/q) — quality grades and regional preferences drive huge variation.
- **Green Chilli** is highly volatile with a wide interquartile range — prices swing significantly by region and season.
- **Tomato, Onion, Potato** have relatively tight distributions, reflecting their status as staples with better-connected supply chains.
- The median lines (orange) show that most commodities cluster in the ₹1,000–3,000/q range, but outlier mandis can see prices 3-4x higher.

---

## 📊 Insight 5: Price Leaders vs Stable Mandis (Onion Case Study)

**Question:** Which mandis show the most price volatility (potential price leaders) vs. which are most stable?

**Why it matters:** Price leader mandis often signal national price trends before other markets react. Stable mandis are better for predictable procurement. This analysis helps traders and policy makers identify which markets to watch.

In [ ]:
onion = df[df['Commodity'] == 'Onion'].copy()
mandi_stats = (
    onion.groupby(['Market', 'State'])
    .agg(
        avg_price=('Modal_Price', 'mean'),
        price_std=('Modal_Price', 'std'),
        records=('Modal_Price', 'count'),
        avg_deviation=('price_vs_state_avg', 'mean'),
    )
    .query('records >= 10')
    .assign(volatility=lambda x: x['price_std'] / x['avg_price'] * 100)
    .sort_values('volatility', ascending=False)
)

top_leaders = mandi_stats.head(10)
bottom_stable = mandi_stats.sort_values('volatility').head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Most volatile
colors1 = plt.cm.Reds(np.linspace(0.4, 0.9, len(top_leaders)))
labels1 = [f"{m}, {s}" for (m, s) in top_leaders.index]
ax1.barh(range(len(top_leaders)), top_leaders['volatility'], color=colors1, height=0.65)
ax1.set_yticks(range(len(top_leaders)))
ax1.set_yticklabels(labels1, fontsize=10)
ax1.set_xlabel('Price Volatility (%)', fontsize=11)
ax1.set_title('Most Volatile Onion Mandis\n(Price Leaders)', fontsize=13, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)
for i, val in enumerate(top_leaders['volatility']):
    ax1.text(val + 0.2, i, f'{val:.1f}%', va='center', fontsize=10, color='#e57373')

# Most stable
colors2 = plt.cm.Greens(np.linspace(0.4, 0.9, len(bottom_stable)))
labels2 = [f"{m}, {s}" for (m, s) in bottom_stable.index]
ax2.barh(range(len(bottom_stable)), bottom_stable['volatility'], color=colors2, height=0.65)
ax2.set_yticks(range(len(bottom_stable)))
ax2.set_yticklabels(labels2, fontsize=10)
ax2.set_xlabel('Price Volatility (%)', fontsize=11)
ax2.set_title('Most Stable Onion Mandis\n(Consistent Pricing)', fontsize=13, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)
for i, val in enumerate(bottom_stable['volatility']):
    ax2.text(val + 0.1, i, f'{val:.1f}%', va='center', fontsize=10, color='#81c784')

plt.suptitle('Onion Market Dynamics: Price Leaders vs Stable Mandis', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/05_price_leaders_vs_stable.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:**
- **Kerala mandis dominate the volatile list** (Chengannur at 49.1%, Ettumanoor at 47.1%). Kerala is far from onion-producing regions, so prices swing wildly with supply disruptions.
- **Haryana and Himachal Pradesh mandis are the most stable** (Solan at 1.2%, Dhand at 1.5%). These are closer to production zones with steady supply.
- The volatility gap is massive: 49% vs 1.2% — a 40x difference. This means Kerala traders face enormous price risk compared to Haryana traders.
- **Actionable insight:** A procurement team should source from stable mandis (Haryana, Himachal) for consistent pricing, but watch volatile mandis (Kerala) as early warning signals for national price movements.

---

## 🎯 Summary of Key Insights

| # | Insight | Key Finding |
|---|---------|-------------|
| 1 | **Price Variance** | Ginger, Leafy Vegetables, and Tomato have the highest pricing inconsistency across mandis (60-100% CV) |
| 2 | **Cheapest States** | Madhya Pradesh is consistently cheapest for all 3 essentials. Kerala/NE states pay 3-10x premiums |
| 3 | **Monthly Trends** | Bhindi prices rising into summer. Tomato/Onion/Potato remain stable — staple supply chains work |
| 4 | **Price Spread** | Banana and Green Chilli have the widest price distributions — biggest arbitrage opportunities |
| 5 | **Market Dynamics** | Kerala mandis are 40x more volatile than Haryana mandis for onion — distance from production = risk |

**These insights power IndiaMandi's RAG pipeline** — when a user asks "where is onion cheapest?", the system retrieves relevant records and Claude synthesizes answers using exactly this kind of analysis.

---
*Analysis by IndiaMandi | Data: data.gov.in | Built as part of IITM Data Science portfolio*